# 걷기 하이브리드 v3 재측정 — GPU 배치 (로컬 CPU 3~4분/프레임 → GPU 수 초)

**내용:** 동영상 3편(862-912)을 1fps 프레임으로 쪼개 **YOLO v3 + v4 rec 하이브리드**(--no_title)로 배치.
결과 zip을 로컬 `walk_grade.py`로 채점 → 판독률 확정 + 검출기 v4 부트스트랩 재료.

**업로드:** `daelim_hybrid_v3.zip` · **GPU(T4) 런타임** · 셀1 후 **[런타임 → 세션 다시 시작]** 필수

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 GPU paddle(>=3.3) + onnxruntime(YOLO)
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q paddleocr onnxruntime
import paddle; print('paddle', paddle.__version__)
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 패키지 업로드
from google.colab import files
up = files.upload()   # daelim_hybrid_v3.zip
!unzip -oq daelim_hybrid_v3.zip -d work/
%cd work
!ls

In [ ]:
# 3) 동영상 → 프레임 (1fps) — 파일명은 ASCII(v1~v3)로 받고 스템은 원본(동영상N)으로 복원
import cv2, os
NAME = {'v1': '동영상1', 'v2': '동영상2', 'v3': '동영상3'}
os.makedirs('frames', exist_ok=True)
for v in sorted(os.listdir('videos')):
    stem = NAME[v.split('.')[0]]
    cap = cv2.VideoCapture(f'videos/{v}')
    fps = cap.get(cv2.CAP_PROP_FPS); step = int(round(fps))
    i = n = 0
    while True:
        ok, fr = cap.read()
        if not ok: break
        if i % step == 0:
            cv2.imwrite(f'frames/{stem}_f{i:05d}.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 95]); n += 1
        i += 1
    cap.release(); print(stem, '→', n, '프레임')

In [ ]:
# 4) 하이브리드 배치 — YOLO v3 검출 + v4 rec 직독 (--no_title = 걷기 실시간 경로)
import glob, subprocess, sys, time
t0 = time.time()
fr_list = sorted(glob.glob('frames/*.jpg'))
for k, p in enumerate(fr_list):
    r = subprocess.run([sys.executable, '-u', 'daelim_yolo_pipeline.py', p,
                        '--catalog', 'catalog_900.csv', '--no_title',
                        '--yolo', 'call_label_yolo3/best.onnx'],
                       capture_output=True, text=True)
    tail = [ln for ln in r.stdout.splitlines() if ln.startswith('[매칭')]
    print(f'[{k+1}/{len(fr_list)}] {p.split("/")[-1]}', tail[-1] if tail else '', flush=True)
    if r.returncode != 0: print(r.stderr[-600:])
print(f'총 {time.time()-t0:.0f}초')

In [ ]:
# 5) 동영상별 투표 합산 (빠른 미리보기 — 공식 채점은 로컬 walk_grade.py)
import json, glob, re
from collections import Counter
vids = {}
for f in glob.glob('out_ondevice/*_f?????_result.json'):
    m = re.search(r'(\d)_f(\d{5})', f)
    if m: vids.setdefault(m.group(1), []).append(f)
for v in sorted(vids):
    votes = Counter()
    for f in sorted(vids[v]):
        for r in json.load(open(f, encoding='utf-8')):
            if r.get('call'): votes[r['call']] += 1
    stab = sum(1 for c, n in votes.items() if n >= 2)
    print(f'동영상{v}: 합산 고유 {len(votes)}권 · 2프레임 안정 {stab}권')

In [ ]:
# 6) 결과 다운로드
!zip -q -r ../daelim_hybrid_v3_results.zip out_ondevice
from google.colab import files
files.download('../daelim_hybrid_v3_results.zip')